In [31]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor

from lineartree import (
    LinearTreeRegressor,
    SobolevForestRegressor,
)
import numpy as np
from sklearn.preprocessing import SplineTransformer
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

In [32]:
class TensorSplineRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, n_knots=6, degree=3, knots="quantile", r_alpha=None):
        self.n_knots = n_knots
        self.degree = degree
        self.knots = knots
        self.r_alpha = r_alpha

    def _basis(self, X, alpha):
        blocks = [
            spline(X[:, j], nu=alpha[j])
            for j, spline in enumerate(self.spline_.bsplines_)
        ]

        basis = blocks[0]
        for block in blocks[1:]:
            basis = np.einsum(
                "ni,nj->nij", basis, block
            ).reshape(len(X), -1)

        return basis

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.n_features_in_ = X.shape[1]

        self.spline_ = SplineTransformer(
            n_knots=self.n_knots,
            degree=self.degree,
            knots=self.knots,
            include_bias=True,
            extrapolation="continue",
        ).fit(X)

        basis = self._basis(X, (0,) * self.n_features_in_)

        if self.r_alpha is None:
            self.model_ = LinearRegression(
                fit_intercept=False
            ).fit(basis, y)
        else:
            self.model_ = Ridge(
                fit_intercept=False,
                alpha=self.r_alpha,
            ).fit(basis, y)

        return self

    def predict(self, X, alpha=None):
        check_is_fitted(self, "model_")
        X = check_array(X)

        alpha = (
            (0,) * self.n_features_in_
            if alpha is None
            else alpha
        )

        if len(alpha) != self.n_features_in_:
            raise ValueError("alpha must have one entry per feature")

        return self._basis(X, alpha) @ self.model_.coef_

In [33]:
def f(x):
    h1 = np.maximum(x[:, 0] - 0.4, 0)
    h2 = np.maximum(x[:, 1] - 0.65, 0)

    return (
        x[:, 0]**2
        + x[:, 1]**2
        + 4 * h1**2 * x[:, 1]
        - 3 * h2**3
    )

def gradient_f(x):
    h1 = np.maximum(x[:, 0] - 0.4, 0)
    h2 = np.maximum(x[:, 1] - 0.65, 0)

    return np.column_stack((
        2 * x[:, 0] + 8 * h1 * x[:, 1],
        2 * x[:, 1] + 4 * h1**2 - 9 * h2**2,
    ))

def hessian_f(x):
    h1 = np.maximum(x[:, 0] - 0.4, 0)
    h2 = np.maximum(x[:, 1] - 0.65, 0)

    hessian = np.empty((len(x), 2, 2))
    hessian[:, 0, 0] = 2 + 8 * x[:, 1] * (x[:, 0] > 0.4)
    hessian[:, 0, 1] = hessian[:, 1, 0] = 8 * h1
    hessian[:, 1, 1] = 2 - 18 * h2

    return hessian


def make_mc(
    seeds=range(100),
    n_train=1_000,
    n_test=10_000,
    sigma=0.2,
):
    mc = {}

    for seed in seeds:
        rng = np.random.default_rng(seed)
        x_train = rng.uniform(size=(n_train, 2))
        x_test = rng.uniform(size=(n_test, 2))

        mc[seed] = (
            x_train,
            f(x_train) + rng.normal(scale=sigma, size=n_train),
            x_test,
            f(x_test),
        )

    return mc

In [34]:
mse_20, mse_11, mse_02 = [[], []], [[], []], [[], []]
mae_20, mae_11, mae_02 = [[], []], [[], []], [[], []]

i = 0
for seed, data in make_mc(seeds=range(20), n_train=1000, n_test=1000).items():
    print(i + 1)
    x_train, y_train, x_test, y_test = data
    tree = LinearTreeRegressor(
        base_estimator=Ridge(alpha=1e-3),
        #base_estimator=LinearRegression(),
        local_degree=3,
        derivative_degree=2,
        max_depth=2,
        min_samples_leaf=30,
        max_bins=25,
        max_features=2,
        random_state=seed,
        derivative_weights=[0.0, 1e-5],
    )

    forest = SobolevForestRegressor(
        estimator=tree,
        n_estimators=500,
        max_samples=0.7,
        n_jobs=-1,
        random_state=seed,
    )

    d = x_train.shape[1]
    n_knots = 3

    knots = np.tile(
        np.linspace(0, 1, n_knots)[:, None],
        (1, d),
    )

    spline = TensorSplineRegressor(
        degree=3,
        knots=knots,
        r_alpha=1e-3,
    ).fit(x_train, y_train)


    hessian_train = hessian_f(x_train)
    gradient_train = gradient_f(x_train)
    hessian_test = hessian_f(x_test)
    gradient_test = gradient_f(x_test)

    deriv = {(1, 0):gradient_train[:, 0], (0, 1):gradient_train[:, 1],
             (2,0): hessian_train[:, 0, 0], (1, 1): hessian_train[:, 1, 0], (0,2): hessian_train[:, 1, 1]}

    forest.fit(x_train, y_train, deriv)

    f_hat_20 = forest.predict(x_test, alpha=(2, 0))
    f_hat_11 = forest.predict(x_test, alpha=(1, 1))
    f_hat_02 = forest.predict(x_test, alpha=(0, 2))

    spline_hat_20 = spline.predict(x_test, alpha=(2, 0))
    spline_hat_11 = spline.predict(x_test, alpha=(1, 1))
    spline_hat_02 = spline.predict(x_test, alpha=(0, 2))



    d20_test = hessian_test[:, 0, 0]
    d11_test = hessian_test[:, 0, 1]
    d02_test = hessian_test[:, 1, 1]

    mse_20[0].append(((f_hat_20 - d20_test) ** 2).mean())
    mse_11[0].append(((f_hat_11 - d11_test) ** 2).mean())
    mse_02[0].append(((f_hat_02 - d02_test) ** 2).mean())
    mae_20[0].append(np.abs(f_hat_20 - d20_test).mean())
    mae_11[0].append(np.abs(f_hat_11 - d11_test).mean())
    mae_02[0].append(np.abs(f_hat_02 - d02_test).mean())

    mse_20[1].append(((spline_hat_20 - d20_test) ** 2).mean())
    mse_11[1].append(((spline_hat_11 - d11_test) ** 2).mean())
    mse_02[1].append(((spline_hat_02 - d02_test) ** 2).mean())
    mae_20[1].append(np.abs(spline_hat_20 - d20_test).mean())
    mae_11[1].append(np.abs(spline_hat_11 - d11_test).mean())
    mae_02[1].append(np.abs(spline_hat_02 - d02_test).mean())

    print('MSE forest:', np.mean(mse_20[0]), 'MAE forest:', np.mean(mae_20[0]))
    print('MSE spline:', np.mean(mse_20[1]), 'MAE spline:', np.mean(mae_20[1]))

    i += 1

1
MSE forest: 3.056412494680755 MAE forest: 1.310421182000306
MSE spline: 8.812167640887088 MAE spline: 2.064242350301423
2
MSE forest: 2.4615049622918455 MAE forest: 1.171582914620216
MSE spline: 7.789572499930667 MAE spline: 2.0533882789087623
3
MSE forest: 2.22182303375325 MAE forest: 1.123652085698563
MSE spline: 10.26223839809775 MAE spline: 2.2731493060665318
4
MSE forest: 2.0683313352205834 MAE forest: 1.1088930863312771
MSE spline: 11.68607670358061 MAE spline: 2.3776088342775217
5
MSE forest: 2.231754907667813 MAE forest: 1.169540291673652
MSE spline: 10.893245468459908 MAE spline: 2.3421032204920404
6
MSE forest: 2.3564177243609747 MAE forest: 1.189976380138147
MSE spline: 11.017126829773076 MAE spline: 2.3201583798777436
7
MSE forest: 2.2627724021329594 MAE forest: 1.1313309305331043
MSE spline: 9.92372400022775 MAE spline: 2.1679217344428756
8
MSE forest: 2.2230661307153885 MAE forest: 1.1257271358461578
MSE spline: 9.211968677068002 MAE spline: 2.088526224786138
9
MSE fore

In [31]:
# derivative weight = 1e-6 / n_samples_in_node
print('MSE_20 forest:', np.mean(mse_20[0]), 'MAE_20 forest:', np.mean(mae_20[0]))
print('MSE_20 spline:', np.mean(mse_20[1]), 'MAE_20 forest:', np.mean(mae_20[1]))
print('MSE_11 forest:', np.mean(mse_11[0]), 'MAE_11 forest:', np.mean(mae_11[0]))
print('MSE_11 spline:', np.mean(mse_11[1]), 'MAE_11 forest:', np.mean(mae_11[1]))
print('MSE_02 forest:', np.mean(mse_02[0]), 'MAE_02 forest:', np.mean(mae_02[0]))
print('MSE_02 spline:', np.mean(mse_02[1]), 'MAE_02 forest:', np.mean(mae_02[1]))

MSE_20 forest: 3.368883101263026 MAE_20 forest: 1.4182703695936185
MSE_20 spline: 8.29407668058565 MAE_20 forest: 2.0088279919277356
MSE_11 forest: 0.7616604107076586 MAE_11 forest: 0.641479602058347
MSE_11 spline: 2.7022880596736707 MAE_11 forest: 1.1230724208227565
MSE_02 forest: 3.982150931819109 MAE_02 forest: 1.3964368976246395
MSE_02 spline: 5.0826772241412455 MAE_02 forest: 1.5411919173189736


In [16]:
# usual cart forest
print('MSE_20 forest:', np.mean(mse_20[0]), 'MAE_20 forest:', np.mean(mae_20[0]))
print('MSE_20 spline:', np.mean(mse_20[1]), 'MAE_20 forest:', np.mean(mae_20[1]))
print('MSE_11 forest:', np.mean(mse_11[0]), 'MAE_11 forest:', np.mean(mae_11[0]))
print('MSE_11 spline:', np.mean(mse_11[1]), 'MAE_11 forest:', np.mean(mae_11[1]))
print('MSE_02 forest:', np.mean(mse_02[0]), 'MAE_02 forest:', np.mean(mae_02[0]))
print('MSE_02 spline:', np.mean(mse_02[1]), 'MAE_02 forest:', np.mean(mae_02[1]))

MSE_20 forest: 3.3653346402886215 MAE_20 forest: 1.4175659228427873
MSE_20 spline: 8.29407668058565 MAE_20 forest: 2.0088279919277356
MSE_11 forest: 0.7619421232983419 MAE_11 forest: 0.641633249267495
MSE_11 spline: 2.7022880596736707 MAE_11 forest: 1.1230724208227565
MSE_02 forest: 3.976530799751682 MAE_02 forest: 1.3954843238649126
MSE_02 spline: 5.0826772241412455 MAE_02 forest: 1.5411919173189736
